# TS1 fit from the complexity summary

Reproduces the calculation behind the first row of
`outputs/complexity_summary.csv`: TS1, no terminal extension, at
`max_edge_length = 1 ×` the mesh thickness median.

The summary reports the best of three stochastic fits (volume ratio nearest
1, then shorter runtime). This notebook does the same three fits and plots
the selected cable. A new run will not match the saved numbers exactly,
because basis optimization uses unseeded ray jitter.

Paths are resolved from the repo root. The kernel cwd can be the repo root
or this notebook's directory.

In [26]:
import logging
from pathlib import Path

import numpy as np

from mascaf import (
    CableFitter,
    FitOptions,
    FitOracleOptions,
    MeshManager,
    SkeletonGraph,
    Validation,
    compute_fit_features,
    suggest_fit_parameters,
    visualize_mesh_cable_3d,
)

logging.basicConfig(level=logging.INFO)

In [27]:
def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "mascaf").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the mascaf repo from the current directory.")


ROOT = repo_root()
MESH_PATH = ROOT / "data" / "mesh" / "processed" / "TS1.obj"
SKELETON_PATH = ROOT / "data" / "mcf_skeletons" / "TS1_qst0.5_mcst5.polylines.txt"
MEL_OVER_THICKNESS = 2.0
N_RUNS = 1

## Load mesh and skeleton

In [28]:
mesh = MeshManager(mesh_path=str(MESH_PATH))
skeleton = SkeletonGraph.from_txt(str(SKELETON_PATH))
features = compute_fit_features(mesh, skeleton)
thickness = float(features.thickness.median)
mel = MEL_OVER_THICKNESS * thickness
print(
    f"TS1: {mesh.mesh.vertices.shape[0]} vertices, "
    f"thickness median={thickness:.6g}, mel={mel:.6g}"
)
print(
    f"Skeleton: {features.skeleton_nodes} nodes, "
    f"{features.n_branches} branches, cyclomatic={features.cyclomatic_number}"
)

INFO:mascaf.mesh:Loaded mesh: 2378 vertices, 4788 faces


TS1: 2378 vertices, thickness median=77.9434, mel=155.887
Skeleton: 376 nodes, 25 branches, cyclomatic=8


## Fit three times and keep the best volume ratio

Same options as `scripts/complexity_analysis.py`: the fit oracle, with
`max_edge_length` overridden to one thickness median. Active resampling is
off. Terminals are not extended (`extend_terminals = 0` in the summary).

In [29]:
opts = FitOracleOptions()
min_len = opts.active_resample_min_over_mel * mel
max_len = opts.active_resample_max_over_mel * mel
diagonal = float(features.bbox_diagonal)
min_frac = min_len / diagonal if diagonal > 0 else 0.0
max_frac = max_len / diagonal if diagonal > 0 else 0.1
min_frac = float(np.clip(min_frac, 1e-6, 1.0))
max_frac = float(np.clip(max_frac, min_frac * 2.0, 1.0))

suggested = suggest_fit_parameters(
    mesh,
    skeleton,
    features=features,
    overrides={
        "max_edge_length": mel,
        "active_resample_min_fraction": min_frac,
        "active_resample_max_fraction": max_frac,
    },
)
for line in suggested.rationale:
    print(f"Oracle: {line}")
print(f"active_resample={suggested.basis_optimizer_options.active_resample}")

options = FitOptions(
    max_edge_length=suggested.max_edge_length,
    basis_optimizer_options=suggested.basis_optimizer_options,
)

do_extend = True

runs = []
for run_index in range(N_RUNS):
    morphology = CableFitter(options).fit(mesh, skeleton)
    if do_extend:
        morphology.extend_terminals()
    validator = Validation(mesh, skeleton, morphology)
    volume = validator.compare_volumes(account_for_overlaps=False)
    area = validator.compare_surface_areas(account_for_overlaps=False)
    volume_overlaps = validator.compare_volumes(account_for_overlaps=True)
    area_overlaps = validator.compare_surface_areas(account_for_overlaps=True)
    record = {
        "run_index": run_index,
        "morphology": morphology,
        "nodes": morphology.number_of_nodes(),
        "volume_ratio": volume["ratio"],
        "volume_relative_error": volume["relative_error"],
        "area_relative_error": area["relative_error"],
        "volume_relative_error_overlaps": volume_overlaps["relative_error"],
        "area_relative_error_overlaps": area_overlaps["relative_error"],
    }
    runs.append(record)
    print(
        f"run {run_index}: {record['nodes']} nodes, "
        f"volume error={record['volume_relative_error']:.4g}, "
        f"area error={record['area_relative_error']:.4g}, "
        f"volume error (overlaps)={record['volume_relative_error_overlaps']:.4g}, "
        f"area error (overlaps)={record['area_relative_error_overlaps']:.4g}"
    )

selected = min(runs, key=lambda rec: (abs(rec["volume_ratio"] - 1.0), rec["run_index"]))
print(
    f"Selected run {selected['run_index']} "
    f"(volume ratio {selected['volume_ratio']:.4g})"
)

INFO:mascaf.fit_oracle:Fit oracle: mel=155.887 (frac=0.05751, mel/t=2), n_rays=12, active_resample=[1e-06, 0.05751]×D
INFO:mascaf.cable_fitting:Starting cable fit with 376 skeleton nodes, 383 skeleton edges, 2378 mesh vertices, and max_edge_length=155.88672959234412
INFO:mascaf.morphology_graph:Resampled morphology: 78 nodes, 84 edges (max_edge_length=155.8867, source 376 nodes / 383 edges, 49 sections)
INFO:mascaf.cable_fitting:Optimizing morphology basis before radius fitting
INFO:mascaf.basis_optimizer:Starting basis optimization...
INFO:mascaf.basis_optimizer:  Input: 78 nodes, 84 edges, total length 8543.6712
INFO:mascaf.basis_optimizer:Phase 0 - Pruning
INFO:mascaf.basis_optimizer:  Removing terminal stubs with length < 85.7628
INFO:mascaf.basis_optimizer:  Pruning done: 78 → 69 nodes (1 rounds)
INFO:mascaf.basis_optimizer:Phase 1 - Snapping: 6 nodes outside mesh


Oracle: thickness-based mel = 1 × SDF median (77.9434) → 77.9434
Oracle: equivalent bbox fraction = 0.02876 (D=2710.45, D/t=34.77, L/t=113.3)
Oracle: Override max_edge_length → 155.887
Oracle: Override basis options: active_resample_max_fraction, active_resample_min_fraction
active_resample=False


INFO:mascaf.basis_optimizer:Phase 2 - Forcing: max 20 iterations
INFO:mascaf.basis_optimizer:  Forcing alpha_s=0.1000 step_scale=0.5000 ray_jitter=0.1000 active_resample=False
INFO:mascaf.basis_optimizer:  Iteration 0: avg movement = 4.138186, centering_error = 6.638379e+02
INFO:mascaf.basis_optimizer:  Iteration 1: avg movement = 3.446325, centering_error = 5.543826e+02
INFO:mascaf.basis_optimizer:  Iteration 2: avg movement = 2.913278, centering_error = 4.692661e+02
INFO:mascaf.basis_optimizer:  Iteration 3: avg movement = 2.882341, centering_error = 4.565154e+02
INFO:mascaf.basis_optimizer:  Iteration 4: avg movement = 2.545677, centering_error = 4.074857e+02
INFO:mascaf.basis_optimizer:  Iteration 5: avg movement = 2.130031, centering_error = 3.403920e+02
INFO:mascaf.basis_optimizer:  Iteration 6: avg movement = 2.084319, centering_error = 3.299883e+02
INFO:mascaf.basis_optimizer:  Iteration 7: avg movement = 2.352390, centering_error = 3.769746e+02
INFO:mascaf.basis_optimizer:  It

run 0: 78 nodes, volume error=1.509, area error=0.4353, volume error (overlaps)=0.2912, area error (overlaps)=0.09882
Selected run 0 (volume ratio 2.509)


## Mesh, skeleton, and selected cable

In [30]:
fig = visualize_mesh_cable_3d(
    mesh,
    skeleton,
    selected["morphology"],
    show_axes=False,
    top_title="TS1 mesh and skeleton",
    bottom_title=f"Cable, run {selected['run_index']}, mel/t={MEL_OVER_THICKNESS:g}",
)
fig

INFO:swctools.geometry:batch_frusta count=84 sides=16 end_caps=False verts=2688 faces=2688


INFO:swctools.geometry:FrustaSet.from_swc_model edges=84 sides=16 end_caps=False
INFO:swctools.viz:plot_model slider=False frusta=84 show_frusta=True show_centroid=True


In [31]:
validator.full_validation()

INFO:mascaf.validation:Validation Results, account_for_overlaps=False:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       70083740.2341
INFO:mascaf.validation:---- Morphology volume: 175818634.8674
INFO:mascaf.validation:---- Ratio:             2.5087
INFO:mascaf.validation:---- Error:             105734894.6333
INFO:mascaf.validation:---- Relative error:    150.87%
INFO:mascaf.validation:-- Surface Area Comparison:
INFO:mascaf.validation:---- Mesh area:         3124839.6923
INFO:mascaf.validation:---- Morphology area:   4484999.9455
INFO:mascaf.validation:---- Ratio:             1.4353
INFO:mascaf.validation:---- Error:             1360160.2532
INFO:mascaf.validation:---- Relative error:    43.53%
INFO:mascaf.validation:Validation Results, account_for_overlaps=True:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       70083740.2341
INFO:mascaf.validation:---- Morphology volume: 90488632.2234
INFO:mascaf.va

{'n_outside_nodes': 5, 'n_crossing_edges': 16, 'n_fully_outside_edges': 0}